# 2. Solve the ink question

Which digit has the highest mean ink, and how much do its images vary?
Recover the answer from small summaries computed by the deployed SuperNodes.

An image's **ink** is the mean of its 784 pixel values, normalized to `[0, 1]`.
We describe the population of image-level ink values for each digit. This is
different from variation between pixels or uncertainty in the mean.


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "notebooks/workshop.py").is_file() and (path / "configs/train.toml").is_file()
)
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import analytics_student, run_analytics

import numpy as np

RUN_ON_FEDERATION = False  # Enable after local checks, login and membership.
FEDERATION = "@charles/flta"


## Choose a local summary

Continue in `work/paris-mnist-fl/mnist_fl/analytics_student.py`:

- `local_moments(images, labels)` returns ten-element `counts`, `sums` and `sumsq` lists
- `merge_moments(reports)` returns the pooled fields, plus ten-element `mean` and `std` lists

`images` is a normalized NumPy array of shape `(N, 784)`. Return population
standard deviations. A digit absent globally has `None` for its mean and
standard deviation. A digit absent locally contributes zeros.

Decide how to recover the global statistics before coding. What would go
wrong if you averaged each node's per-digit mean? The
[analytics guide](../docs/analytics.md#2-which-digit-uses-the-most-ink) has a hint
and the complete input/output contract.


In [ ]:
student = analytics_student(ROOT)
images = np.stack([np.full(784, ink) for ink in [0.2, 0.8, 0.8, 0.5]])
labels = np.array([2, 2, 2, 7])
reports = [
    student.local_moments(images[:1], labels[:1]),
    student.local_moments(images[1:], labels[1:]),
]
for report in reports:
    assert set(report) == {"counts", "sums", "sumsq"}
    assert all(type(values) is list and len(values) == 10 for values in report.values())
    assert all(type(value) is int for value in report["counts"])
result = student.merge_moments(reports)
assert set(result) == {"counts", "sums", "sumsq", "mean", "std"}
assert all(type(values) is list and len(values) == 10 for values in result.values())
assert sum(result["counts"]) == 4 and result["counts"][2] == 3
assert np.isclose(result["sums"][2], 1.8)
assert np.isclose(result["sumsq"][2], 1.32)
assert np.isclose(result["mean"][2], 0.6)
assert np.isclose(result["std"][2], np.sqrt(0.08))
assert np.isclose(result["std"][7], 0.0)
assert result["mean"][0] is None and result["std"][0] is None
print("Your moments functions passed the uneven-count check")


The two sites above contain different numbers of digit-2 images. Their
unweighted means would not recover the pooled answer. Also check a node with
no images, a digit whose images all have the same ink, and two images with
different pixels but identical mean ink.


In [ ]:
student = analytics_student(ROOT)
empty = student.local_moments(np.empty((0, 784)), np.array([], dtype=int))
assert empty["counts"] == [0] * 10
assert empty["sums"] == empty["sumsq"] == [0.0] * 10
constant = student.local_moments(np.full((3, 784), 0.4), np.array([5, 5, 5]))
constant_result = student.merge_moments([empty, constant])
assert constant_result["counts"][5] == 3
assert np.isclose(constant_result["mean"][5], 0.4)
assert np.isclose(constant_result["std"][5], 0.0, atol=1e-8)
# Different pixels can still give two images exactly the same mean ink.
mixed = np.stack([np.tile([0.0, 1.0], 392), np.full(784, 0.5)])
mixed_result = student.merge_moments([student.local_moments(mixed, np.array([4, 4]))])
assert mixed_result["counts"][4] == 2
assert np.isclose(mixed_result["sums"][4], 1.0)
assert np.isclose(mixed_result["sumsq"][4], 0.5)
assert np.isclose(mixed_result["std"][4], 0.0, atol=1e-8)
print("Empty, constant-ink and image-level variation checks passed")


## Ask the deployed data

Make a prediction for the highest-mean-ink digit. Enable `RUN_ON_FEDERATION`
when ready, then run both partition schemes. These are new student-code runs
over the same 6,000 training images; neither publishes the shared report.


In [ ]:
iid_run = run_analytics(
    ROOT, "moments", "iid", execute=RUN_ON_FEDERATION, federation=FEDERATION
)


In [ ]:
skew_run = run_analytics(
    ROOT, "moments", "label-skew", execute=RUN_ON_FEDERATION, federation=FEDERATION
)


In [ ]:
if iid_run is None or skew_run is None:
    print("Comparison pending: complete both deployed runs first")
else:
    iid, skew = iid_run["summary"], skew_run["summary"]
    assert sum(iid["counts"]) == sum(skew["counts"]) == 6000
    assert iid["counts"] == skew["counts"]
    for field in ["sums", "sumsq", "mean", "std"]:
        np.testing.assert_allclose(
            np.asarray(iid[field], dtype=float), np.asarray(skew[field], dtype=float),
            rtol=1e-10, atol=1e-12, equal_nan=True,
        )
    print("digit | count | mean ink | population std")
    for digit in range(10):
        print(f"{digit:>5} | {iid['counts'][digit]:>5} | "
              f"{iid['mean'][digit]:.6f} | {iid['std'][digit]:.6f}")
    print("Complete deployed moments agree within tolerance")


## Keep your observations

| Observation | Your result |
| --- | --- |
| Prediction | |
| IID run ID | |
| Label-skew run ID | |
| Highest-mean-ink digit | |
| Its count, mean and population standard deviation | |
| An uneven-count case you checked | |

**Checkpoint:** your local and global moment functions pass the small checks,
and the completed deployed results agree across partition schemes. Explain
why the mean of site means can fail even when the sites have equal total sizes.

## Prepare the agent question

Wait for one approved complete moments report to be published. The nodes will
hold copies of that same global report. In the next exercise, compare those
copies and return one; they are not additive contributions.

Start reading `work/paris-federation-agent/federation_agent/guided.py`. What
capability should the agent expose? Which arguments should it accept?

Next: [the federation agent](03_federation_agent.ipynb), or choose an
[ink-statistics extension](../docs/extensions.md#after-the-ink-statistics).
